In [8]:
%pip install pandas duckdb

895.33s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


In [9]:
import duckdb
import pandas as pd
import os

In [10]:
if os.path.exists('ecommerce.duckdb'):
    # We try to connect in-memory first to ensure we aren't locking anything
    duckdb.connect().close()

# 1. Connect (This will create a FRESH, valid database file)
con = duckdb.connect('ecommerce.duckdb')

# Load CSVs into tables
con.execute("CREATE OR REPLACE TABLE orders AS SELECT * FROM read_csv_auto('data/List of Orders.csv');")
con.execute("CREATE OR REPLACE TABLE order_details AS SELECT * FROM read_csv_auto('data/Order Details.csv');")
con.execute("CREATE OR REPLACE TABLE sales_target AS SELECT * FROM read_csv_auto('data/Sales target.csv');")

# Quick check: List all tables
con.sql("SHOW TABLES").show()

┌───────────────┐
│     name      │
│    varchar    │
├───────────────┤
│ order_details │
│ orders        │
│ sales_target  │
└───────────────┘



## What is wrong with the data?

In [14]:
def q(sql: str):
    return con.sql(sql).df()

In [18]:
q("DESCRIBE orders;")


,column_name,column_type,null,key,default,extra
0,Order ID,VARCHAR,YES,None,None,None
1,Order Date,DATE,YES,None,None,None
2,CustomerName,VARCHAR,YES,None,None,None
3,State,VARCHAR,YES,None,None,None
4,City,VARCHAR,YES,None,None,None


In [17]:
q("DESCRIBE order_details;")


,column_name,column_type,null,key,default,extra
0,Order ID,VARCHAR,YES,None,None,None
1,Amount,DOUBLE,YES,None,None,None
2,Profit,DOUBLE,YES,None,None,None
3,Quantity,BIGINT,YES,None,None,None
4,Category,VARCHAR,YES,None,None,None
5,Sub-Category,VARCHAR,YES,None,None,None


In [19]:
q("DESCRIBE sales_target;")


,column_name,column_type,null,key,default,extra
0,Month of Order Date,VARCHAR,YES,None,None,None
1,Category,VARCHAR,YES,None,None,None
2,Target,DOUBLE,YES,None,None,None


In [ ]:
# Row counts
q("""
SELECT 'orders' AS table, COUNT(*) AS rows FROM orders
UNION ALL
SELECT 'order_details', COUNT(*) FROM order_details
UNION ALL
SELECT 'sales_target', COUNT(*) FROM sales_target;
""")

,table,rows
0,orders,560
1,order_details,1500
2,sales_target,36


They all have null values, lets see how many, and what to do with them


In [23]:
q("""
SELECT
  SUM(CASE WHEN "Order ID" IS NULL THEN 1 ELSE 0 END) AS null_order_id,
  SUM(CASE WHEN "Order Date" IS NULL THEN 1 ELSE 0 END) AS null_order_date
FROM orders;
""")

,null_order_id,null_order_date
0,60.0,60.0


Teniendo en cuenta que hay 560, igual deberiamos mirar en los que son null order id

In [24]:
# Duplicate Order IDs in the orders header table
q("""
SELECT "Order ID", COUNT(*) AS n
FROM orders
WHERE "Order ID" IS NOT NULL
GROUP BY 1
HAVING COUNT(*) > 1;
""")


,Order ID,n


In [ ]:
# Details that do not match any order header (Orphan lines)
q("""
SELECT COUNT(*) AS orphan_lines
FROM order_details d
LEFT JOIN orders o
  ON d."Order ID" = o."Order ID"
WHERE o."Order ID" IS NULL;
""")


,orphan_lines
0,0


In [27]:
# White space issues, like "Furniture" or " Furniture"
q("""
SELECT DISTINCT State
FROM orders
WHERE State IS NOT NULL AND State <> TRIM(State);
""")

,State
0,Kerala


We must trim text field Kerala

## Cleaning pipeline

In [30]:
con.execute("CREATE SCHEMA IF NOT EXISTS stg;")

In [31]:
con.execute("CREATE SCHEMA IF NOT EXISTS clean;")

In [ ]:
# stg.orders (one row per order, fixed strings)

con.execute("""
CREATE OR REPLACE VIEW stg.orders AS
SELECT
  TRIM("Order ID") AS order_id,
  "Order Date"::DATE AS order_date,
  TRIM(CustomerName) AS customer_name,
  TRIM(State) AS state,
  TRIM(City) AS city
FROM orders
WHERE "Order ID" IS NOT NULL;  -- drops blank rows safely
""")